# ORM | QuerySets | field lookup

### Table of Contents

- [Introduction](#introduction)
  - [Create](#create)
  - [Read](#read)
  - [Update](#update)
  - [Delete](#delete)
  - [Complex lookups with Q objects](#)

## Introduction

ORM stands for Object Relational Mapper. Django ORM is based on QuerySets.

Let's explore basic CRUD (Create, Read, Update, & Delete) Operations

## Create

### How to create an object?

Import the respected models, to add a new post in the Post model with title, slug, body and author fields.

```py
>>> from django.contrib.auth.models import User
>>> from blog.models import Post
>>> user = User.objects.get(username='admin')
>>> post = Post(title="New Post",
                slug="new-post",
                body="post body.",
                author=user)
>>> post.save()
```
**Notes:**

1. Don't forget to call the save() method in the post instance explicitly
2. User.**objects**.get() -> this objects is called Manager, check out this article about [How to create a custom model manager?](3-Model-Manager.ipynb)

Another way to create a post is to use create() method, as follow:

```py
>>> Post.objects.create(title="One more post",
                    slug="one-more-post",
                    body="Post body.",
                    author=user)
```
**Note:** For above method we don't need to call save() method

In certain situations, you might need to fetch an object from the database or create it if it's absent, we can use get_or_create() method

```py
>>> user, created = User.objects.get_or_create(username='user2')
```

above method returns a tuple contains an object retrived or Boolean indicate whether a new object was created.

## Read

Import all the necessary models like

```py
from blog.models import Post
```

### How to get / retrieve all objects?

```py
>>> all_posts = Post.objects.all() # QuerySets are not executed because the django QuerySets are lazy.
>>> all_posts # Output: [<Post: New Post>, <Post: One more post>]

# or we can directly execute the QuerySets

>>> Post.objects.all() # Output: [<Post: New Post>, <Post: One more post>]
```

### How to get / retrieve single object?

```py
>>> Post.objects.get(id=1)
```

### Using filter() method to retrieving objects

> To filter a QuerySet, you can use the filter() method of the manager. This method allows you to specify the content of a SQL WHERE clause by using field lookups.

`>>> Post.objects.filter(title="New Post")`

Above QuerySet will return all posts with the exact title New Post.

#### Do You Know?

You can get the equivalent SQL Query of the django querysets with query method, check the example below

```py
>>> posts = Post.objects.filter(title="New Post")
>>> print(posts.query)

# output

SELECT "blog_post"."id", "blog_post"."title", "blog_post"."slug", "blog_post"."author_id", "blog_post"."body", "blog_post"."publish", "blog_post"."created", "blog_post"."updated", "blog_post"."status" FROM "blog_post" WHERE "blog_post"."title" = New Post ORDER BY "blog_post"."publish" DESC
```

### Using field lookups in filter() method

> Two underscores are used to define the lookup type, with the format field__lookup.

**Get exact match**

`>>> Post.object.filter(id__exact=1)`

When no specific lookup type is provided, the lookup type is assumed to be `__exact` following, below is same as above query 

`>>> Post.objects.filter(id=1)`

**Chaining additional lookup**

```py
>>> Post.objects.filter(publish__date__gt=date(2024, 1, 1))
```

**Lookup related objects fields**

`>>> Post.objects.filter(author__username='admin')`

**Lookup related objects fields with Chaining**

`>>> Post.objects.filter(author__username__starstwith='ad')`

#### Chaining Filters

```py
>>> Post.objects.filter(publish__year=2024, author__username='admin')
# or
>>> Post.objects.filter(publish__year=2024).filter(author__username='admin')
```

#### Excluding Objects

> You can exclude certain results from your QuerySet by using the exclude() method of the manager.

`>>> Post.objects.filter(publish__year=2024).exclude(title__startswith='Why')`

### Ordering Objects

`>>> Post.objects.order_by('title')` or by descending `Post.objects.order_by('-title')`

**Ordering multiple fields**

`>>> Post.objects.order_by('author', 'title')`

**Order randomly**

`>>> Post.objects.order_by('?')`

### Limiting Objects

> By using a subset of Python’s array-slicing syntax, You can limit a QuerySet to a certain number of results

`>>> Post.objects.all()[:5]` **Note:** This translates to a SQL LIMIT 5 clause

`>>> Post.objects.all()[3:6]` **Note:** This translates to a SQL OFFSET 3 LIMIT 6 clause

**Randomly Retrieve an object** : To retrieve the first object of posts in random order

`>>> Post.objects.order_by('?')[0]`

### Counting objects

`>>> Post.objects.filter(id_lt=3).count() # output: 2` **Note:** It translates to a SELECT COUNT(*) SQL statement

### Checking if an object exists

`>>> Post.objects.filter(title__startswith='Why').exists() # output: False`

## List of Field Lookups

| lookups | Description |
| ------- | ----------- |
| __exact | Retrun exact value which matches |
| __iexact | Retrun exact value which matches with case-insensitive |
| __contains | Retrun value which matches with given query, it act as SQL LIKE operator |
| __icontains | Retrun value which matches with case-insensitive, it act as a WHERE ... LIKE %Django% |
| __in | check for a given iterable (often a list, tuple, or another QuerySet object) with the in lookup |
| __gt | show the greater than lookup, it act as WHERE ID > 3 |
| __gte | show the greater than and equal lookup, it act as WHERE ID >= 3 |
| __lt | show the less than lookup, it act as WHERE ID < 3 |
| __gte | show the less than and equal lookup, it act as WHERE ID <= 3 |
| __startswith | A case-sensitive starts-with lookup can be performed |
| __istartswith | A case-insensitive starts-with lookup can be performed |
| __endswith | A case-sensitive ends-with lookup can be performed |
| __iendswith | A case-insensitive ends-with lookup can be performed |
| __date | filter a DateField or DateTimeField field by exact date |
| __year | filter a DateField or DateTimeField field by year |
| __month | filter a DateField or DateTimeField field by month |
| __day | filter a DateField or DateTimeField field by day |

## Update

### How to update an single object?

Get the object instance with get() method and modify the filed value and don't forget to call the save() method.

```py
>>> from blog.models import Post
>>> post = Post.object.get(id=1)
>>> post.title = "New Title"
>>> post.save()
```

This time, the save() method performs an SQL UPDATE statement

## Delete

### How to delete an single object?

If you want to delete an object, first get() the object and use delete() method.

```py
post = Post.objects.get(id=1)
post.delete()
```

**Note:** that deleting objects will also delete any dependent relationships for ForeignKey objects defined with on_delete set to CASCADE.

## Complex lookups with Q objects

> Field lookups using filter() are joined with a SQL AND operator. For example, filter(field1='foo ', field2='bar') will retrieve objects where field1 is foo and field2 is bar. If you need to build more complex queries, such as queries with OR statements, you can use Q objects. A Q object allows you to encapsulate a collection of field lookups. You can compose statements by 
combining Q objects with the & (and), | (or), and ^ (xor) operators.

```py
>>> from django.db.models import Q
>>> starts_who = Q(title__istartswith='who')
>>> starts_why = Q(title__istartswith='why')
>>> Post.objects.filter(starts_who | starts_why)
```

visit for more at https://docs.djangoproject.com/en/5.0/topics/db/queries/#complex-lookups-with-q-objects

## QuerySets are only evaluated in the following cases:
- The first time you iterate over them
- When you slice them, for instance, Post.objects.all()[:3]
- When you pickle or cache them
- When you call repr() or len() on them
- When you explicitly call list() on them
- When you test them in a statement, such as bool(), or, and, or if